# Edge Client Notebook  Client 0
## BoT-IoT Federated Learning  Edge Device Training

This notebook implements a single federated learning client. Each client holds a private data shard, trains the shared FT-Transformer model locally for a fixed number of epochs per round, then returns the updated weights to the server, never the raw data.

This notebook is parameterised for Client 0 and must be duplicated and reconfigured (changing `client_id`) for each additional client. All clients run in parallel and connect to the same server address.

---


### Cell 1  Client Definitions: Model, Training Loop, and Flower Client

**Purpose:** Defines the local model, training and evaluation routines, and the Flower client class that plugs into the federated communication protocol.

**FTTransformer (local model)**
The Feature Tokeniser Transformer embeds each of the 36 network-traffic features into a dedicated 128-dimensional vector via a per-feature linear projection (`feature_embeddings`), prepends a learnable CLS token, passes the resulting sequence through a 6-layer Transformer encoder, and reads off the CLS token's final representation for classification. This architecture respects the heterogeneous nature of tabular data: each feature gets its own embedding space rather than being forced into a shared one.

**`get_parameters` / `set_parameters`**
These helpers serialise and deserialise the model's state dictionary into the flat list of NumPy arrays that Flower requires for network transmission.

**`local_train(model, train_loader, config, device, local_epochs)`**
Runs the local training loop for the specified number of epochs. It uses the AdamW optimiser with the learning rate and weight decay injected by the server via `config`, and cross-entropy loss as the objective. At the end it returns the average training loss, the weighted F1 score, and the number of training samples. The sample count is used by the server to weight this client's contribution to the aggregate.

**`local_evaluate(model, val_loader, device)`**
Evaluates the model on the client's held-out validation split without updating any weights. It computes loss, accuracy, weighted F1, precision, recall, AUC (via one-vs-rest multi-class ROC), and the confusion matrix.

**`BotIoTClient`**
The Flower client class with two methods Flower calls each round:

- `fit`: Receives the latest global weights from the server, loads them into the local model, runs `local_train`, evaluates on the training set, and appends a metrics record to a per-client JSON file (`client_N_metrics.json`). Returns the updated weights, the sample count, and summary metrics.
- `evaluate`: Receives the global weights, runs `local_evaluate` on the validation split, logs the results to the same JSON file, and returns the loss and all metrics to the server. This is the per-client validation signal the server uses to track generalisation progress.


In [ ]:
"""
client.py
---------
Flower FL client for the BoT-IoT FT-Transformer.

Each client loads its own data shard and trains locally.

Usage (in a separate terminal for each client):
    python client.py --client_id 0 --data_dir ./client_data --server_address localhost:8080
    python client.py --client_id 1 --data_dir ./client_data --server_address localhost:8080
    python client.py --client_id 2 --data_dir ./client_data --server_address localhost:8080
    ... (duplicate for as many clients as you sharded)

Notes:
    - Start the server FIRST, then start the clients.
    - Each client reads client_{id}_X.npy and client_{id}_y.npy
    - Local training uses a train/val split within the client's shard.
"""
from tqdm import tqdm

import argparse
import json
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from collections import OrderedDict
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix

import flwr as fl
from flwr.common import NDArrays, Scalar


# ─────────────────────────────────────────────────────────────
# FT-Transformer (must match server)
# ─────────────────────────────────────────────────────────────

class FTTransformer(nn.Module):
    def __init__(self, n_features, d_model=128, n_heads=8,
                 n_layers=6, d_ffn=512, n_classes=5, dropout=0.1):
        super().__init__()
        self.feature_embeddings = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(n_features)
        ])
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_ffn, dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_model, n_classes)
        )

    def forward(self, x):
        batch_size = x.size(0)
        embeddings = [self.feature_embeddings[i](x[:, i:i+1])
                      for i in range(x.size(1))]
        embeddings = torch.stack(embeddings, dim=1)
        cls = self.cls_token.expand(batch_size, -1, -1)
        embeddings = torch.cat([cls, embeddings], dim=1)
        embeddings = self.transformer(embeddings)
        return self.head(embeddings[:, 0])


# ─────────────────────────────────────────────────────────────
# Parameter utilities
# ─────────────────────────────────────────────────────────────

def get_parameters(model: nn.Module) -> NDArrays:
    return [val.cpu().numpy() for _, val in model.state_dict().items()]


def set_parameters(model: nn.Module, parameters: NDArrays) -> None:
    params_dict = zip(model.state_dict().keys(), parameters)
    state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
    model.load_state_dict(state_dict, strict=True)


# ─────────────────────────────────────────────────────────────
# Local training & evaluation
# ─────────────────────────────────────────────────────────────

def local_train(model, train_loader, config, device, local_epochs):
    """Train the model for a given number of local epochs."""
    #local_epochs = int(config.get("local_epochs", 3))
    lr           = float(config.get("lr", 1e-4))
    weight_decay = float(config.get("weight_decay", 1e-5))

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    criterion = nn.CrossEntropyLoss()
    model.train()
    model.to(device)

    all_preds, all_labels = [], []
    total_loss = 0.0
    total_samples = 0

    from tqdm import tqdm

    # inside local_train, replace the epoch loop with:
    for epoch in range(local_epochs):
        pbar = tqdm(train_loader, desc=f"Client {model.__class__.__name__} Epoch {epoch+1}/{local_epochs}", leave=False)
        for X_batch, y_batch in pbar:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out  = model(X_batch)
            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()

            total_loss    += loss.item() * len(y_batch)
            total_samples += len(y_batch)
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

            pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / total_samples
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    return avg_loss, f1, total_samples


def local_evaluate(model, val_loader, device):
    """Evaluate the model on local val/test data."""
    criterion = nn.CrossEntropyLoss()
    model.eval()
    model.to(device)

    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            out  = model(X_batch)
            loss = criterion(out, y_batch)
            total_loss    += loss.item() * len(y_batch)
            total_samples += len(y_batch)
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    avg_loss = total_loss / total_samples
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)


    # replace the return line with:
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall    = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    cm        = confusion_matrix(all_labels, all_preds).tolist()

    # AUC needs probabilities : add this inside torch.no_grad() loop:
    all_probs.extend(F.softmax(out, dim=1).cpu().numpy().tolist())

    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='weighted')
    except:
        auc = 0.0

    return float(avg_loss), float(accuracy), float(f1), float(precision), float(recall), float(auc), cm, total_samples

    #return float(avg_loss), float(accuracy), float(f1), total_samples


# ─────────────────────────────────────────────────────────────
# Flower Client
# ─────────────────────────────────────────────────────────────

class BotIoTClient(fl.client.NumPyClient):

    def __init__(self, client_id, model, train_loader, val_loader, device, local_epochs, data_dir):
        self.client_id    = client_id
        self.model        = model
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.device       = device
        self.local_epochs = local_epochs
        self.data_dir     = data_dir

    def get_parameters(self, config) -> NDArrays:
        return get_parameters(self.model)

    def set_parameters(self, parameters: NDArrays) -> None:
        set_parameters(self.model, parameters)

    # -------------------------------------------------
    # FIT (TRAIN)
    # -------------------------------------------------
    def fit(
        self,
        parameters: NDArrays,
        config: Dict[str, Scalar]
    ) -> Tuple[NDArrays, int, Dict[str, Scalar]]:

        self.set_parameters(parameters)

        round_num = int(config.get("current_round", -1))

        print(f"\n[Client {self.client_id}] Training (local_epochs={self.local_epochs}) ...")

        train_loss, train_f1, n_samples = local_train(
            self.model, self.train_loader, config, self.device, self.local_epochs
        )

        print(f"[Client {self.client_id}] Train Loss: {train_loss:.4f} | F1: {train_f1:.4f}")

        # ---- Compute full metrics on TRAIN set (same metrics as evaluation)
        loss, acc, f1, precision, recall, auc, cm, _ = local_evaluate(
            self.model, self.train_loader, self.device
        )

        train_metrics = {
            "round": round_num,
            "phase": "train",
            "loss": loss,
            "accuracy": acc,
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "auc": auc,
            "confusion_matrix": cm
        }

        history_path = os.path.join(self.data_dir, f'client_{self.client_id}_metrics.json')

        history = []
        if os.path.exists(history_path):
            with open(history_path) as f:
                history = json.load(f)

        history.append(train_metrics)

        with open(history_path, "w") as f:
            json.dump(history, f, indent=2)

        return (
            self.get_parameters(config={}),
            n_samples,
            {"train_loss": float(train_loss), "train_f1": float(train_f1)}
        )

    # -------------------------------------------------
    # EVALUATE
    # -------------------------------------------------
    def evaluate(self, parameters, config):

        self.set_parameters(parameters)

        loss, acc, f1, precision, recall, auc, cm, n_samples = local_evaluate(
            self.model, self.val_loader, self.device
        )

        round_num = int(config.get("current_round", -1))

        eval_metrics = {
            "round": round_num,
            "phase": "eval",
            "loss": loss,
            "accuracy": acc,
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "auc": auc,
            "confusion_matrix": cm
        }

        history_path = os.path.join(self.data_dir, f'client_{self.client_id}_metrics.json')

        history = []
        if os.path.exists(history_path):
            with open(history_path) as f:
                history = json.load(f)

        history.append(eval_metrics)

        with open(history_path, "w") as f:
            json.dump(history, f, indent=2)

        print(
            f"[Client {self.client_id}] Round {round_num} | Loss: {loss:.4f} | "
            f"Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}"
        )

        return loss, n_samples, {
            "accuracy": acc,
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "auc": auc
        }

### Cell 2 Client Execution

**Purpose:** Loads Client 0's data shard, builds the model and data loaders, and connects to the server.

The cell reads `client_0_X.npy` and `client_0_y.npy` from the shared `client_data` directory. The data is wrapped in a PyTorch `TensorDataset` and split 80/20 into training and validation subsets using a seeded random generator, ensuring the split is identical across repeated runs.

The `BotIoTClient` instance is passed to `fl.client.start_client`, which opens a gRPC connection to `localhost:9090` and waits for instructions from the server.

**Observed training trajectory (Client 0, 5 rounds of 3 local epochs):**

| Round | Training Loss | Training F1 | Eval Accuracy | Eval F1 |
|-------|-------------|-------------|---------------|---------|
| 1     | 0.0598      | 0.9838      | 0.9454        | 0.9304  |
| 2     | 0.0140      | 0.9954      | 0.9703        | 0.9704  |
| 3     | 0.0089      | 0.9973      | 0.9892        | 0.9889  |
| 4     | 0.0074      | 0.9977      | 0.9917        | 0.9913  |
| 5     | 0.0071      | 0.9979      | 0.9906        | 0.9905  |

Client 0 holds 48,052 samples  the smallest shard. Even so, the model converges quickly and achieves greater than 99% F1 on the local validation set by Round 3. The slight fluctuation between rounds is normal and reflects the effect of integrating updates from four other clients with different data distributions each time the global model is refreshed.


In [2]:
# ─────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────
# Run in Notebook
# ─────────────────────────────────────────────────────────────
from tqdm import tqdm

# Set parameters directly
client_id = 0  
data_dir = '/content/client_data'
server_address = 'localhost:9090' 
val_split = 0.2
batch_size = 64
local_epochs = 3

# Load metadata
meta_path = os.path.join(data_dir, 'meta.json')
with open(meta_path) as f:
    meta = json.load(f)
n_features = meta['n_features']
n_classes = meta['n_classes']

# Load client data
X_path = os.path.join(data_dir, f'client_{client_id}_X.npy')
y_path = os.path.join(data_dir, f'client_{client_id}_y.npy')
X = np.load(X_path)
y = np.load(y_path)

print(f"\n{'='*60}")
print(f"  FL Client {client_id} – BoT-IoT FT-Transformer")
print(f"{'='*60}")
print(f"  Samples    : {len(X):,}")
print(f"  Features   : {n_features}")
print(f"  Classes    : {n_classes} {meta['classes']}")
print(f"  Server     : {server_address}")
print(f"{'='*60}\n")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Build dataset and split
dataset = TensorDataset(torch.FloatTensor(X), torch.LongTensor(y))
n_val = int(len(dataset) * val_split)
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42 + client_id)
)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=batch_size * 2, shuffle=False, num_workers=0)

print(f"Train samples : {n_train:,}")
print(f"Val samples   : {n_val:,}\n")

# Build model
model = FTTransformer(n_features=n_features, n_classes=n_classes).to(device)

# Build Flower client
client = BotIoTClient(
    client_id=client_id,
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    local_epochs=local_epochs,
    data_dir=data_dir
)

# Connect to server
print(f"Connecting to server at {server_address} ...\n")
fl.client.start_client(
    server_address=server_address,
    client=client.to_client()
)


  FL Client 0 – BoT-IoT FT-Transformer
  Samples    : 48,052
  Features   : 36
  Classes    : 5 ['DDoS', 'DoS', 'Normal', 'Reconnaissance', 'Theft']
  Server     : localhost:9090

Train samples : 38,442
Val samples   : 9,610



	Instead, use the `flower-supernode` CLI command to start a SuperNode as shown below:

		$ flower-supernode --insecure --superlink='<IP>:<PORT>'

	To view all available options, run:

		$ flower-supernode --help

	Using `start_client()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
	Instead, use the `flower-supernode` CLI command to start a SuperNode as shown below:

		$ flower-supernode --insecure --superlink='<IP>:<PORT>'

	To view all available options, run:

		$ flower-supernode --help

	Using `start_client()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
DEBUG:flwr:Using InMemoryObjectStore
DEBUG:flwr:Opened insecure gRPC connection (no certificates were passed)
DEBUG:flwr:ChannelConnectivity.IDLE
DEBUG:flwr:ChannelConnectivity.READY


Connecting to server at localhost:9090 ...



INFO :      
INFO:flwr:
INFO :      Received: train message 54e24c19-0ae0-4967-bd37-a9c7fb342ece
INFO:flwr:Received: train message 54e24c19-0ae0-4967-bd37-a9c7fb342ece



[Client 0] Training (local_epochs=3) ...


...


[Client 0] Training (local_epochs=3) ...


[Client 0] Train Loss: 0.0071 | F1: 0.9979


INFO :      Sent reply
INFO:flwr:Sent reply
INFO :      
INFO:flwr:
INFO :      Received: evaluate message e7cc2466-be90-407e-83a3-86e801204320
INFO:flwr:Received: evaluate message e7cc2466-be90-407e-83a3-86e801204320
INFO :      Sent reply
INFO:flwr:Sent reply


[Client 0] Round -1 | Loss: 0.0252 | Acc: 0.9906 | F1: 0.9905 | AUC: 0.0000


INFO :      
INFO:flwr:
INFO :      Received: reconnect message 359135a2-6893-49f7-871e-3e9a29e766b2
INFO:flwr:Received: reconnect message 359135a2-6893-49f7-871e-3e9a29e766b2
DEBUG:flwr:gRPC channel closed
INFO :      Disconnect and shut down
INFO:flwr:Disconnect and shut down
